Análise estrututal das imagens de exemplo

---

Bibliotecas

In [ ]:
import cv2

import matplotlib.pyplot as plt
import networkx          as nx
import numpy             as np

from collections import deque


---

Carregando a imagem do Josefran

In [ ]:
josefran_image = cv2.imread("../../assets/josefran.jpg")
josefran_image = cv2.cvtColor(josefran_image, cv2.COLOR_BGR2GRAY)

josefran_image.shape

In [ ]:
plt.imshow(josefran_image, cmap='gray')
plt.show()

Redimensionando para $64\times64$

In [ ]:
josefran_image = cv2.resize(josefran_image, (64, 64), cv2.INTER_LINEAR)
josefran_image.shape

Visualização da imagem final

In [ ]:
plt.imshow(josefran_image, cmap='gray')
plt.show()

---

Carregando a imagem de um ave

In [ ]:
ave_image = cv2.imread("../../assets/ave.jpg")
ave_image = cv2.cvtColor(ave_image, cv2.COLOR_BGR2GRAY)

ave_image.shape

In [ ]:
plt.imshow(ave_image, cmap='gray')
plt.show()

Redimensionando para $64\times64$

In [ ]:
ave_image = cv2.resize(ave_image, (64, 64), cv2.INTER_LINEAR)
ave_image.shape

In [ ]:
plt.imshow(ave_image, cmap='gray')
plt.show()

Visualização da imagem final

In [ ]:
plt.imshow(ave_image, cmap='gray')
plt.show()

---

Carregando a imagem de meme

In [ ]:
meme_image = cv2.imread("../../assets/meme.jpg")
meme_image = cv2.cvtColor(meme_image, cv2.COLOR_BGR2GRAY)

meme_image.shape

In [ ]:
plt.imshow(meme_image, cmap='gray')
plt.show()

Redimensionando para $64\times64$

In [ ]:
meme_image = cv2.resize(meme_image, (64, 64), cv2.INTER_LINEAR)
meme_image.shape

In [ ]:
plt.imshow(meme_image, cmap='gray')
plt.show()

Visualização da imagem final

In [ ]:
plt.imshow(meme_image, cmap='gray')
plt.show()

---

Algoritmo para construir a representação em grafos de uma imagem

In [ ]:
def construir_grafo_imagem(matriz_pixels, vizinhanca=4, limiar=10):
    altura, largura = matriz_pixels.shape
    grafo           = { (i, j): [] for i in range(altura) for j in range(largura) }
    
    # Definir deslocamentos com base na vizinhança
    if vizinhanca == 4:
        direcoes = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    elif vizinhanca == 8:
        direcoes = [(-1, 0), (1, 0), (0, -1), (0, 1), 
                    (-1, -1), (-1, 1), (1, -1), (1, 1)]
    else:
        raise ValueError("A vizinhança deve ser 4 ou 8.")

    for i in range(altura):
        for j in range(largura):
            u = (i, j)
            intensidade_u = matriz_pixels[i, j]
            
            for di, dj in direcoes:
                ni, nj = i + di, j + dj
                
                if 0 <= ni < altura and 0 <= nj < largura:
                    v = (ni, nj)
                    intensidade_v = matriz_pixels[ni, nj]
                    
                    if abs(int(intensidade_u) - int(intensidade_v)) <= limiar:
                        grafo[u].append(v)
                        
    return grafo

---

Algoritmo para encontrar as componentes conexas

In [ ]:
def encontrar_componentes_dfs(grafo):
    visitados = set()
    componentes = []

    for pixel_inicial in grafo:
        if pixel_inicial not in visitados:
            # Inicia uma nova componente conexa
            pixels_componente = []
            fila = deque([pixel_inicial])
            visitados.add(pixel_inicial)

            while fila:
                u = fila.popleft()
                pixels_componente.append(u)

                for v in grafo[u]:
                    if v not in visitados:
                        visitados.add(v)
                        fila.append(v)

            componentes.append({
                'tamanho': len(pixels_componente),
                'pixels': pixels_componente
            })

    # Ordena da maior componente para a menor
    componentes.sort(key=lambda c: c['tamanho'], reverse=True)

    # Atribui um ID sequencial (0 = maior componente)
    for idx, comp in enumerate(componentes):
        comp['id'] = idx

    return componentes

---

Algoritmo para encontrar pontes e articulações

In [ ]:
def encontrar_pontes_e_articulacoes(grafo):
    """
    Identifica pontes e vértices de articulação via DFS iterativa e low[].
    """
    tin = {}
    low = {}
    visitado = set()
    pontes = set()
    articulacoes = set()
    timer = 0

    for raiz in grafo:
        if raiz in visitado:
            continue

        visitado.add(raiz)
        timer += 1
        tin[raiz] = low[raiz] = timer
        filhos_raiz = 0

        # Cada item da pilha: [u, pai, iterador_vizinhos]
        pilha = [[raiz, None, iter(grafo.get(raiz, []))]]

        while pilha:
            u, pai, vizinhos = pilha[-1]

            try:
                v = next(vizinhos)
                if v == pai:
                    continue

                if v in visitado:
                    if tin[v] < low[u]:
                        low[u] = tin[v]
                else:
                    if pai is None:
                        filhos_raiz += 1

                    visitado.add(v)
                    timer += 1
                    tin[v] = low[v] = timer
                    pilha.append([v, u, iter(grafo.get(v, []))])

            except StopIteration:
                pilha.pop()
                if pilha:
                    p = pilha[-1][0]
                    avo = pilha[-1][1]

                    if low[u] < low[p]:
                        low[p] = low[u]

                    # Aresta (p, u) é ponte se low[u] > tin[p]
                    if low[u] > tin[p]:
                        pontes.add((p, u) if p < u else (u, p))

                    # Vértice não-raiz p é articulação se low[u] >= tin[p]
                    if avo is not None and low[u] >= tin[p]:
                        articulacoes.add(p)

        # Raiz da árvore DFS é articulação se tiver 2 ou mais filhos
        if filhos_raiz >= 2:
            articulacoes.add(raiz)

    return pontes, articulacoes

---

Algoritmo para a análise estrututal

In [ ]:
def analise_estrutural_grafo(grafo):
    """
    Extrai as métricas estruturais de um grafo representado como {u: [v1, v2, ...]}.
    """
    num_vertices = len(grafo)

    arestas_unicas = set()
    soma_graus = 0

    for u, vizinhos in grafo.items():
        soma_graus += len(vizinhos)
        for v in vizinhos:
            aresta = (u, v) if u < v else (v, u)
            arestas_unicas.add(aresta)

    num_arestas = len(arestas_unicas)
    grau_medio = (soma_graus / num_vertices) if num_vertices > 0 else 0.0

    componentes = encontrar_componentes_dfs(grafo)
    num_componentes = len(componentes)
    tamanho_maior_componente = len(componentes[0]) if componentes else 0

    pontes, articulacoes = encontrar_pontes_e_articulacoes(grafo)

    return {
        "num_vertices": num_vertices,
        "num_arestas": num_arestas,
        "num_componentes": num_componentes,
        "grau_medio": grau_medio,
        "tamanho_maior_componente": tamanho_maior_componente,
        "num_pontes": len(pontes),
        "num_articulacoes": len(articulacoes),
        "componentes": componentes,
        "pontes": pontes,
        "articulacoes": articulacoes,
    }

In [ ]:
def imprimir_analise_estrutural(metricas, rotulo="Grafo"):
    print(f"=== Análise Estrutural — {rotulo} ===")
    print(f"Número de vértices:          {metricas['num_vertices']}")
    print(f"Número de arestas:           {metricas['num_arestas']}")
    print(f"Número de componentes:       {metricas['num_componentes']}")
    print(f"Grau médio:                  {metricas['grau_medio']:.4f}")
    print(f"Tamanho da maior componente: {metricas['tamanho_maior_componente']}")
    print(f"Número de pontes:            {metricas['num_pontes']}")
    print(f"Número de articulações:      {metricas['num_articulacoes']}")

---

Gerando os grafos

In [ ]:
josefran_graph_10l  = construir_grafo_imagem(josefran_image, vizinhanca = 4, limiar  = 10)
josefran_graph_100l = construir_grafo_imagem(josefran_image, vizinhanca = 4, limiar = 100)
josefran_graph_200l = construir_grafo_imagem(josefran_image, vizinhanca = 4, limiar = 200)

In [ ]:
ave_graph_10l = construir_grafo_imagem(ave_image, vizinhanca = 4, limiar  = 10)
ave_graph_100l = construir_grafo_imagem(ave_image, vizinhanca = 4, limiar = 100)
ave_graph_200l = construir_grafo_imagem(ave_image, vizinhanca = 4, limiar = 200)

In [ ]:
meme_graph_10l = construir_grafo_imagem(meme_image, vizinhanca = 4, limiar  = 10)
meme_graph_100l = construir_grafo_imagem(meme_image, vizinhanca = 4, limiar = 100)
meme_graph_200l = construir_grafo_imagem(meme_image, vizinhanca = 4, limiar = 200)

---

Análise estrututal

In [59]:
import pandas as pd

# 1. Lista com os rótulos e os grafos já construídos
grafos_analise = [
    ("Josefran — Limiar 10",  josefran_graph_10l),
    ("Josefran — Limiar 100", josefran_graph_100l),
    ("Josefran — Limiar 200", josefran_graph_200l),
    ("Meme — Limiar 10",      meme_graph_10l),
    ("Meme — Limiar 100",     meme_graph_100l),
    ("Meme — Limiar 200",     meme_graph_200l),
    ("Ave — Limiar 10",       ave_graph_10l),
    ("Ave — Limiar 100",      ave_graph_100l),
    ("Ave — Limiar 200",      ave_graph_200l),
]

# 2. Extrai as métricas de cada grafo
linhas = []
for rotulo, grafo in grafos_analise:
    metricas = analise_estrutural_grafo(grafo)
    linhas.append({
        "Grafo": rotulo,
        "Vértices": metricas["num_vertices"],
        "Arestas": metricas["num_arestas"],
        "Componentes": metricas["num_componentes"],
        "Grau Médio": round(metricas["grau_medio"], 4),
        "Maior Componente": metricas["tamanho_maior_componente"],
        "Pontes": metricas["num_pontes"],
        "Articulações": metricas["num_articulacoes"],
    })

# 3. Cria o DataFrame definindo o nome do grafo como índice (linhas)
df_analise = pd.DataFrame(linhas).set_index("Grafo")

# Exibe a tabela formatada no notebook
df_analise

,Vértices,Arestas,Componentes,Grau Médio,Maior Componente,Pontes,Articulações
Grafo,,,,,,,
Josefran — Limiar 10,4096,5270,626,2.5732,3,749,564
Josefran — Limiar 100,4096,7780,7,3.7988,3,37,34
Josefran — Limiar 200,4096,8061,1,3.9360,3,0,0
Meme — Limiar 10,4096,4554,946,2.2236,3,986,674
Meme — Limiar 100,4096,7778,2,3.7979,3,17,16
Meme — Limiar 200,4096,8060,1,3.9355,3,0,0
Ave — Limiar 10,4096,6426,417,3.1377,3,346,237
Ave — Limiar 100,4096,7769,11,3.7935,3,26,25
Ave — Limiar 200,4096,8043,1,3.9272,3,1,1
